# Web Scraper Toolkit — Demo

This notebook is a guided walkthrough of the toolkit: run a scraper live, export the results, and do a quick analysis pass with `pandas`.

**Two ways to use this notebook:**
1. **Live mode** (default) — Section 1 calls the real scrapers over the network. Requires internet access and `pip install -r ../requirements.txt`.
2. **Offline mode** — If you have no network access right now (e.g. a locked-down sandbox), skip to Section 3, which loads the pre-generated snapshots in `../samples/` instead. The rest of the notebook works identically either way, since it only cares about having a `pandas.DataFrame` of records.

## 1. Run a scraper live

This cell imports the toolkit modules directly (no need to install the package — the notebook lives one level below the project root) and runs the product scraper against `books.toscrape.com` for a couple of pages.

In [ ]:
import sys
sys.path.append("..")  # so `import scraper` etc. resolves to the project root

from scraper import ProductScraper, NewsScraper, JobScraper

with ProductScraper() as scraper:
    products = scraper.scrape(pages=2)

print(f"Scraped {len(products)} products")
products[:3]

## 2. Export the results

Same `exporter.py` functions the CLI (`main.py`) uses under the hood.

In [ ]:
from exporter import export_records

written = export_records(products, name="products", output_dir="../output")
written

## 3. Load into pandas for a quick look

This cell reads from `../samples/` (pre-generated, committed snapshots) so the rest of the notebook renders correctly even without network access. Point `pd.read_csv` at `../output/products.csv` instead if you want to analyze your own fresh run from Section 1.

In [ ]:
import pandas as pd

df_products = pd.read_csv("../samples/products.csv")
df_news = pd.read_csv("../samples/news.csv")
df_jobs = pd.read_csv("../samples/jobs.csv")

df_products.head(8)

In [ ]:
print("Average book price: £%.2f" % df_products["price"].mean())
print("Average book rating: %.1f / 5" % df_products["rating"].mean())
df_products.sort_values("price", ascending=False)[["name", "price", "rating"]]

In [ ]:
# Which HN story has the highest comments-to-points ratio (i.e. sparked
# the most discussion relative to its score)?
df_news["engagement_ratio"] = (df_news["comments_count"] / df_news["points"]).round(3)
df_news.sort_values("engagement_ratio", ascending=False)[["title", "points", "comments_count", "engagement_ratio"]]

In [ ]:
df_jobs[["title", "company", "location"]]

## 4. Run the full CLI pipeline

The same thing this notebook just did, in one command from the project root:

```bash
python main.py --scraper all --format csv json
```

See `README.md` for the full list of CLI flags (`--pages`, `--output-dir`, `--log-level`, etc.).